In [1]:
import duckdb
con = duckdb.connect()

con.sql("""
    SELECT *
    FROM read_csv_auto('../data/raw/*.csv.gz', union_by_name=true)
    LIMIT 5
""")

[Sortie supprimée : lignes de transactions individuelles — voir README, section Protection des données]

In [2]:
con.sql("""
    CREATE VIEW dvf AS
    SELECT * FROM read_csv_auto('../data/raw/*.csv.gz', union_by_name=true)
""")

In [3]:
con.sql("SELECT COUNT(*) FROM dvf")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       409358 │
└──────────────┘

In [4]:
con.sql("SELECT COUNT(DISTINCT id_mutation) FROM dvf")

┌─────────────────────────────┐
│ count(DISTINCT id_mutation) │
│            int64            │
├─────────────────────────────┤
│                      158391 │
└─────────────────────────────┘

In [5]:
con.sql("""
    SELECT nature_mutation, COUNT(*)
    FROM dvf
    GROUP BY nature_mutation
    ORDER BY COUNT(*) DESC
""")

┌────────────────────────────────────┬──────────────┐
│          nature_mutation           │ count_star() │
│              varchar               │    int64     │
├────────────────────────────────────┼──────────────┤
│ Vente                              │       357964 │
│ Vente en l'état futur d'achèvement │        46327 │
│ Echange                            │         2821 │
│ Vente terrain à bâtir              │         1102 │
│ Adjudication                       │         1003 │
│ Expropriation                      │          141 │
└────────────────────────────────────┴──────────────┘

In [6]:
con.sql("""
    SELECT COUNT(*)
    FROM dvf
    WHERE valeur_fonciere IS NULL
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1473 │
└──────────────┘

In [7]:
con.sql("""
    SELECT n, COUNT(*) AS mutations
    FROM (SELECT id_mutation, COUNT(*) AS n FROM dvf GROUP BY id_mutation)
    GROUP BY n ORDER BY n
""")

┌───────┬───────────┐
│   n   │ mutations │
│ int64 │   int64   │
├───────┼───────────┤
│     1 │     49395 │
│     2 │     58099 │
│     3 │     30099 │
│     4 │      9378 │
│     5 │      3182 │
│     6 │      2545 │
│     7 │      1246 │
│     8 │      1049 │
│     9 │       623 │
│    10 │       458 │
│     · │         · │
│     · │         · │
│     · │         · │
│   201 │         1 │
│   247 │         2 │
│   288 │         1 │
│   308 │         1 │
│   391 │         1 │
│   413 │         1 │
│   436 │         1 │
│   459 │         1 │
│   494 │         1 │
│  1202 │         1 │
└───────┴───────────┘
      130 rows     
     (20 shown)     

In [8]:
con.sql("""
    SELECT id_mutation, COUNT(*) AS n
    FROM dvf GROUP BY id_mutation ORDER BY n DESC LIMIT 5
""")

[Sortie supprimée : lignes de transactions individuelles — voir README, section Protection des données]

In [9]:
con.sql("""
    SELECT id_mutation, valeur_fonciere, type_local, surface_reelle_bati, nombre_pieces_principales, id_parcelle
    FROM dvf WHERE id_mutation = '2025-370737'
""")

[Sortie supprimée : lignes de transactions individuelles — voir README, section Protection des données]

In [10]:
con.sql("""
    SELECT id_mutation, COUNT(*) FILTER (WHERE type_local IN ('Maison', 'Appartement')) AS n_logements
    FROM dvf
    GROUP BY id_mutation
""")

[Sortie supprimée : lignes de transactions individuelles — voir README, section Protection des données]

In [11]:
con.sql("""
    SELECT id_mutation,
       COUNT(*) FILTER (WHERE type_local IN ('Maison', 'Appartement')) AS n_logements
    FROM dvf
    GROUP BY id_mutation
    HAVING COUNT(*) FILTER (WHERE type_local IN ('Maison', 'Appartement')) = 1
""")

[Sortie supprimée : lignes de transactions individuelles — voir README, section Protection des données]

In [12]:
con.sql("""
    SELECT COUNT(*) FROM (
        SELECT id_mutation
        FROM dvf
        GROUP BY id_mutation
        HAVING COUNT(*) FILTER (WHERE type_local IN ('Maison', 'Appartement')) = 1
    )
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        95768 │
└──────────────┘

In [13]:
con.sql("""
    SELECT COUNT(*) FROM (
        SELECT id_mutation
        FROM dvf
        GROUP BY id_mutation
        HAVING COUNT(*) FILTER (WHERE type_local IN ('Maison', 'Appartement')) = 0
    )
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        50976 │
└──────────────┘

In [14]:
con.sql("""
WITH mutations_valides AS (
    SELECT id_mutation
    FROM dvf
    WHERE nature_mutation = 'Vente'
      AND valeur_fonciere IS NOT NULL
    GROUP BY id_mutation
    HAVING COUNT(*) FILTER (WHERE type_local IN ('Maison', 'Appartement')) = 1
)
SELECT COUNT(*) FROM mutations_valides
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        93493 │
└──────────────┘

In [15]:
con.sql("""
WITH mutations_valides AS (
    SELECT id_mutation
    FROM dvf
    WHERE nature_mutation = 'Vente'
      AND valeur_fonciere IS NOT NULL
    GROUP BY id_mutation
    HAVING COUNT(*) FILTER (WHERE type_local IN ('Maison', 'Appartement')) = 1
),
ventes AS (
    SELECT
        d.id_mutation,
        ANY_VALUE(d.valeur_fonciere) AS prix,
        SUM(d.surface_reelle_bati) FILTER (WHERE type_local IN ('Maison', 'Appartement')) AS surface,
        ANY_VALUE(d.date_mutation)  AS date_vente,
        ANY_VALUE(d.code_commune)   AS code_commune,
        ANY_VALUE(d.nom_commune)    AS nom_commune,
        ANY_VALUE(d.type_local) FILTER (WHERE type_local IN ('Maison', 'Appartement')) AS type_bien,
        ANY_VALUE(d.nombre_pieces_principales) FILTER (WHERE type_local IN ('Maison', 'Appartement')) AS pieces
    FROM dvf d
    JOIN mutations_valides USING (id_mutation)
    GROUP BY d.id_mutation
)
SELECT * FROM ventes LIMIT 10
""")

[Sortie supprimée : lignes de transactions individuelles — voir README, section Protection des données]

In [16]:
con.sql("""
CREATE OR REPLACE TABLE ventes_propres AS
WITH mutations_valides AS (
    SELECT id_mutation
    FROM dvf
    WHERE nature_mutation = 'Vente'
      AND valeur_fonciere IS NOT NULL
    GROUP BY id_mutation
    HAVING COUNT(*) FILTER (WHERE type_local IN ('Maison', 'Appartement')) = 1
)
SELECT
    d.id_mutation,
    ANY_VALUE(d.valeur_fonciere) AS prix,
    CAST(SUM(d.surface_reelle_bati)
         FILTER (WHERE d.type_local IN ('Maison', 'Appartement')) AS INTEGER) AS surface,
    ANY_VALUE(d.date_mutation)  AS date_vente,
    ANY_VALUE(d.code_commune)   AS code_commune,
    ANY_VALUE(d.nom_commune)    AS nom_commune,
    ANY_VALUE(d.type_local)
        FILTER (WHERE d.type_local IN ('Maison', 'Appartement')) AS type_bien,
    ANY_VALUE(d.nombre_pieces_principales)
        FILTER (WHERE d.type_local IN ('Maison', 'Appartement')) AS pieces
FROM dvf d
JOIN mutations_valides USING (id_mutation)
GROUP BY d.id_mutation
""")

In [17]:
con.sql("""
SELECT
    COUNT(*) AS total,
    COUNT(*) FILTER (WHERE surface IS NULL) AS sans_surface,
    COUNT(*) FILTER (WHERE surface = 0)     AS surface_zero,
    MIN(surface) AS surface_min,
    MAX(surface) AS surface_max
FROM ventes_propres
""")

┌───────┬──────────────┬──────────────┬─────────────┬─────────────┐
│ total │ sans_surface │ surface_zero │ surface_min │ surface_max │
│ int64 │    int64     │    int64     │    int32    │    int32    │
├───────┼──────────────┼──────────────┼─────────────┼─────────────┤
│ 93493 │            1 │            0 │           1 │         880 │
└───────┴──────────────┴──────────────┴─────────────┴─────────────┘

In [18]:
con.sql("""
SELECT
    quantile_cont(surface, 0.001) AS p001,
    quantile_cont(surface, 0.01)  AS p01,
    quantile_cont(surface, 0.05)  AS p05,
    quantile_cont(surface, 0.50)  AS mediane,
    quantile_cont(surface, 0.95)  AS p95,
    quantile_cont(surface, 0.99)  AS p99,
    quantile_cont(surface, 0.999) AS p999
FROM ventes_propres
""")

┌────────┬────────┬────────┬─────────┬────────┬────────┬────────────────────┐
│  p001  │  p01   │  p05   │ mediane │  p95   │  p99   │        p999        │
│ double │ double │ double │ double  │ double │ double │       double       │
├────────┼────────┼────────┼─────────┼────────┼────────┼────────────────────┤
│   13.0 │   18.0 │   26.0 │    69.0 │  148.0 │  201.0 │ 327.01800000001094 │
└────────┴────────┴────────┴─────────┴────────┴────────┴────────────────────┘

In [19]:
con.sql("""
SELECT
    COUNT(*) FILTER (WHERE surface < 9)  AS moins_9,
    COUNT(*) FILTER (WHERE surface < 15) AS moins_15,
    COUNT(*) FILTER (WHERE surface < 20) AS moins_20,
    COUNT(*) FILTER (WHERE surface > 400) AS plus_400
FROM ventes_propres
""")

┌─────────┬──────────┬──────────┬──────────┐
│ moins_9 │ moins_15 │ moins_20 │ plus_400 │
│  int64  │  int64   │  int64   │  int64   │
├─────────┼──────────┼──────────┼──────────┤
│       9 │      167 │     1499 │       29 │
└─────────┴──────────┴──────────┴──────────┘

In [20]:
con.sql("""
SELECT
    quantile_cont(prix / surface, 0.001) AS p001,
    quantile_cont(prix / surface, 0.01)  AS p01,
    quantile_cont(prix / surface, 0.25)  AS q1,
    quantile_cont(prix / surface, 0.50)  AS mediane,
    quantile_cont(prix / surface, 0.75)  AS q3,
    quantile_cont(prix / surface, 0.99)  AS p99,
    quantile_cont(prix / surface, 0.999) AS p999,
    MIN(prix / surface) AS mini,
    MAX(prix / surface) AS maxi
FROM ventes_propres
WHERE surface BETWEEN 9 AND 400
""")

┌────────────────────┬────────┬────────────────────┬──────────┬────────────────────┬───────────────────┬────────────────────┬──────────────────────┬────────────────────┐
│        p001        │  p01   │         q1         │ mediane  │         q3         │        p99        │        p999        │         mini         │        maxi        │
│       double       │ double │       double       │  double  │       double       │      double       │       double       │        double        │       double       │
├────────────────────┼────────┼────────────────────┼──────────┼────────────────────┼───────────────────┼────────────────────┼──────────────────────┼────────────────────┤
│ 15.013348416289594 │  500.0 │ 2257.1428571428573 │ 2859.375 │ 3573.0337078651687 │ 7165.398856190479 │ 14718.250464646517 │ 0.005847953216374269 │ 232662.85714285713 │
└────────────────────┴────────┴────────────────────┴──────────┴────────────────────┴───────────────────┴────────────────────┴──────────────────────┴──

In [21]:
con.sql("""
SELECT
    COUNT(*) AS nb_groupes,
    COUNT(*) FILTER (WHERE n < 30) AS petits_groupes,
    SUM(n) FILTER (WHERE n < 30) AS ventes_concernees
FROM (
    SELECT code_commune, type_bien, COUNT(*) AS n
    FROM ventes_propres
    WHERE surface BETWEEN 9 AND 400
    GROUP BY code_commune, type_bien
)
""")

┌────────────┬────────────────┬───────────────────┐
│ nb_groupes │ petits_groupes │ ventes_concernees │
│   int64    │     int64      │      int128       │
├────────────┼────────────────┼───────────────────┤
│        776 │            495 │              4488 │
└────────────┴────────────────┴───────────────────┘

In [22]:
con.sql("""
CREATE OR REPLACE TABLE ventes_finales AS
WITH base AS (
    SELECT *, prix / surface AS prix_m2
    FROM ventes_propres
    WHERE surface BETWEEN 9 AND 400
),
seuils_dept AS (
    SELECT type_bien,
           quantile_cont(prix_m2, 0.01) AS p01_dept,
           quantile_cont(prix_m2, 0.99) AS p99_dept
    FROM base GROUP BY type_bien
),
seuils_commune AS (
    SELECT code_commune, type_bien, COUNT(*) AS n,
           quantile_cont(prix_m2, 0.01) AS p01_com,
           quantile_cont(prix_m2, 0.99) AS p99_com
    FROM base GROUP BY code_commune, type_bien
)
SELECT b.*,
       CASE WHEN sc.n >= 30 THEN sc.p01_com ELSE sd.p01_dept END AS seuil_bas,
       CASE WHEN sc.n >= 30 THEN sc.p99_com ELSE sd.p99_dept END AS seuil_haut
FROM base b
JOIN seuils_commune sc USING (code_commune, type_bien)
JOIN seuils_dept    sd USING (type_bien)
""")

In [23]:
con.sql("""
SELECT COUNT(*) AS total,
       COUNT(*) FILTER (WHERE prix_m2 < seuil_bas OR prix_m2 > seuil_haut) AS hors_bornes
FROM ventes_finales
""")

┌───────┬─────────────┐
│ total │ hors_bornes │
│ int64 │    int64    │
├───────┼─────────────┤
│ 93454 │        2276 │
└───────┴─────────────┘

In [24]:
con.sql("""
CREATE OR REPLACE TABLE ventes_finales AS
SELECT id_mutation, prix, surface, prix_m2, date_vente,
       code_commune, nom_commune, type_bien, pieces,
       EXTRACT(YEAR FROM date_vente) AS annee
FROM ventes_finales
WHERE prix_m2 BETWEEN seuil_bas AND seuil_haut
""")

In [25]:
con.sql("""
SELECT type_bien,
       COUNT(*) AS n,
       ROUND(MIN(prix_m2)) AS mini,
       ROUND(quantile_cont(prix_m2, 0.5)) AS mediane,
       ROUND(MAX(prix_m2)) AS maxi
FROM ventes_finales
GROUP BY type_bien
""")

┌─────────────┬───────┬────────┬─────────┬─────────┐
│  type_bien  │   n   │  mini  │ mediane │  maxi   │
│   varchar   │ int64 │ double │ double  │ double  │
├─────────────┼───────┼────────┼─────────┼─────────┤
│ Maison      │ 38490 │   10.0 │  2783.0 │ 11793.0 │
│ Appartement │ 52688 │  279.0 │  2923.0 │  7388.0 │
└─────────────┴───────┴────────┴─────────┴─────────┘

In [26]:
con.sql("""
SELECT COUNT(*) FILTER (WHERE prix_m2 < 100)  AS moins_100,
       COUNT(*) FILTER (WHERE prix_m2 < 300)  AS moins_300,
       COUNT(*) FILTER (WHERE prix_m2 < 500)  AS moins_500,
       COUNT(*) FILTER (WHERE prix_m2 < 800)  AS moins_800
FROM ventes_finales
""")

┌───────────┬───────────┬───────────┬───────────┐
│ moins_100 │ moins_300 │ moins_500 │ moins_800 │
│   int64   │   int64   │   int64   │   int64   │
├───────────┼───────────┼───────────┼───────────┤
│        14 │        98 │       402 │      1331 │
└───────────┴───────────┴───────────┴───────────┘

In [27]:
con.sql("""
SELECT nom_commune, type_bien, prix, surface, ROUND(prix_m2) AS prix_m2, date_vente
FROM ventes_finales
ORDER BY prix_m2
LIMIT 15
""")

[Sortie supprimée : lignes de transactions individuelles — voir README, section Protection des données]

In [28]:
con.sql("""
CREATE OR REPLACE TABLE ventes_finales AS
SELECT * FROM ventes_finales WHERE prix_m2 >= 500
""")

In [29]:
con.sql("""
SELECT type_bien, COUNT(*) AS n,
       ROUND(MIN(prix_m2)) AS mini,
       ROUND(quantile_cont(prix_m2, 0.5)) AS mediane,
       ROUND(MAX(prix_m2)) AS maxi
FROM ventes_finales
GROUP BY type_bien
""")

┌─────────────┬───────┬────────┬─────────┬─────────┐
│  type_bien  │   n   │  mini  │ mediane │  maxi   │
│   varchar   │ int64 │ double │ double  │ double  │
├─────────────┼───────┼────────┼─────────┼─────────┤
│ Maison      │ 38118 │  500.0 │  2794.0 │ 11793.0 │
│ Appartement │ 52658 │  500.0 │  2923.0 │  7388.0 │
└─────────────┴───────┴────────┴─────────┴─────────┘

In [30]:
con.sql("""
SELECT annee,
       type_bien,
       COUNT(*) AS nb_ventes,
       ROUND(quantile_cont(prix_m2, 0.5)) AS prix_m2_median,
       ROUND(quantile_cont(surface, 0.5)) AS surface_mediane
FROM ventes_finales
GROUP BY annee, type_bien
ORDER BY type_bien, annee
""")

┌───────┬─────────────┬───────────┬────────────────┬─────────────────┐
│ annee │  type_bien  │ nb_ventes │ prix_m2_median │ surface_mediane │
│ int64 │   varchar   │   int64   │     double     │     double      │
├───────┼─────────────┼───────────┼────────────────┼─────────────────┤
│  2021 │ Appartement │     12036 │         2848.0 │            55.0 │
│  2022 │ Appartement │     12478 │         2969.0 │            55.0 │
│  2023 │ Appartement │      9530 │         2957.0 │            55.0 │
│  2024 │ Appartement │      8425 │         2872.0 │            55.0 │
│  2025 │ Appartement │     10189 │         2952.0 │            55.0 │
│  2021 │ Maison      │      9202 │         2724.0 │            97.0 │
│  2022 │ Maison      │      8645 │         2906.0 │            96.0 │
│  2023 │ Maison      │      6743 │         2889.0 │            95.0 │
│  2024 │ Maison      │      6189 │         2747.0 │            96.0 │
│  2025 │ Maison      │      7339 │         2745.0 │            96.0 │
└─────

In [31]:
con.sql("SELECT annee, COUNT(*) FROM ventes_finales GROUP BY annee ORDER BY annee")
con.sql("SELECT MIN(date_mutation), MAX(date_mutation) FROM dvf")

┌────────────────────┬────────────────────┐
│ min(date_mutation) │ max(date_mutation) │
│        date        │        date        │
├────────────────────┼────────────────────┤
│ 2021-01-04         │ 2025-12-31         │
└────────────────────┴────────────────────┘

In [32]:
con.sql("""
CREATE OR REPLACE TABLE agg_commune_annee AS
SELECT
    code_commune,
    ANY_VALUE(nom_commune) AS nom_commune,
    annee,
    type_bien,
    COUNT(*) AS nb_ventes,
    ROUND(quantile_cont(prix_m2, 0.5)) AS prix_m2_median,
    ROUND(AVG(prix_m2)) AS prix_m2_moyen,
    ROUND(quantile_cont(prix, 0.5)) AS prix_median,
    ROUND(quantile_cont(surface, 0.5)) AS surface_mediane
FROM ventes_finales
GROUP BY code_commune, annee, type_bien
HAVING COUNT(*) >= 5
""")

In [33]:
con.sql("SELECT COUNT(*) FROM agg_commune_annee")
con.sql("SELECT SUM(nb_ventes) FROM agg_commune_annee")

┌────────────────┐
│ sum(nb_ventes) │
│     int128     │
├────────────────┤
│          87757 │
└────────────────┘

In [34]:
con.sql("""
COPY agg_commune_annee
TO '../data/processed/agg_commune_annee.parquet' (FORMAT PARQUET)
""")

In [35]:
con.sql("""
COPY (
    SELECT annee, type_bien,
           COUNT(*) AS nb_ventes,
           ROUND(quantile_cont(prix_m2, 0.5)) AS prix_m2_median,
           ROUND(quantile_cont(prix, 0.5)) AS prix_median
    FROM ventes_finales
    GROUP BY annee, type_bien
) TO '../data/processed/agg_departement.parquet' (FORMAT PARQUET)
""")

In [36]:
con.sql("""
SELECT annee, type_bien, nb_ventes, prix_m2_median
FROM agg_commune_annee
WHERE code_commune = 31555
ORDER BY type_bien, annee
""")

┌───────┬─────────────┬───────────┬────────────────┐
│ annee │  type_bien  │ nb_ventes │ prix_m2_median │
│ int64 │   varchar   │   int64   │     double     │
├───────┼─────────────┼───────────┼────────────────┤
│  2021 │ Appartement │      7780 │         3210.0 │
│  2022 │ Appartement │      8091 │         3304.0 │
│  2023 │ Appartement │      6123 │         3305.0 │
│  2024 │ Appartement │      5469 │         3182.0 │
│  2025 │ Appartement │      6672 │         3250.0 │
│  2021 │ Maison      │      1411 │         3766.0 │
│  2022 │ Maison      │      1354 │         4000.0 │
│  2023 │ Maison      │      1019 │         3955.0 │
│  2024 │ Maison      │       946 │         3674.0 │
│  2025 │ Maison      │      1154 │         3710.0 │
└───────┴─────────────┴───────────┴────────────────┘
  10 rows                                4 columns